# Stage 1 SBI Pretraining

Notebook này chỉ dành cho **Stage 1**. Mục tiêu là pretrain model bằng `real + SBI fake` để model học **generic forgery cues** trước khi sang deepfake thật.

Ý tưởng của Stage 1:
- dùng ảnh real làm nguồn gốc
- tạo SBI fake từ real
- chỉ bật `LoRA` + `classifier`
- tắt `adapter` và `MoE router`
- mục tiêu là học boundary / blending / artifact cues tổng quát


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/cs415/moe-deepfake'
REPO_URL = 'https://github.com/kdnehihi/SBI-MoE-Deepfake-Detection.git'
PROJECT_DIR = '/content/SBI-MoE-Deepfake-Detection'


In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd {PROJECT_DIR}


## Install dependencies


In [ ]:
!pip install -q timm facenet-pytorch opencv-python==4.10.0.84 PyYAML Pillow scikit-learn


## Build Stage 1 dataset

Stage 1 lấy real từ `Celeb-DF train + FF++ train`, rồi sinh SBI fake tương ứng để tạo tập `50% real / 50% SBI fake`.


In [ ]:
!python data/prepare_stage_datasets.py \
  --stage stage1 \
  --celebdf-root "$DATA_ROOT/data/processed/celebdf" \
  --ffpp-root "$DATA_ROOT/data/processed/ffpp_generalization" \
  --output-root "$DATA_ROOT/data/stages/stage1_sbi" \
  --overwrite


In [ ]:
!ls -lh "$DATA_ROOT/data/stages/stage1_sbi"/*.jsonl
!find "$DATA_ROOT/data/stages/stage1_sbi" -maxdepth 2 -type d | sort


## Train Stage 1

Mặc định Stage 1:
- bật `LoRA`
- tắt `adapter`
- tắt `MoE router`
- train classifier như paper-flow của repo hiện tại

Nếu cần chạy nhanh trước, hạ `--epochs` xuống `3` hoặc `5`.


In [ ]:
!python -u train_stage1.py \
  --dataset-root "$DATA_ROOT/data/stages/stage1_sbi" \
  --batch-size 16 \
  --epochs 5 \
  --num-workers 4 \
  --image-size 224 \
  --device cuda


## Notes

- checkpoint sẽ được lưu vào `outputs/stage1_last.pt` trong Colab runtime
- nếu muốn giữ lâu dài, copy checkpoint đó sang Drive sau khi train xong
- trong lúc train, trainer giờ sẽ in thêm usage của `LoRA` và `Adapter` experts nếu model có MoE active
